# MDFF + GSEFE Algorithms

This notebook gives the **algorithm-level implementation** for `GSEFE` and `MDFF` so you can build the `.py` files yourself. It is written for the LBMS-SAM style architecture described in your attached note.

The attached reference says: GSEFE takes the **raw image directly**, MDFF takes **GA1-GA4 intermediate features**, and the final fusion combines SAM mask-side features with MDFF and GSEFE outputs. [file:18]


## GSEFE algorithm

### Purpose
GSEFE means **Gabor-Sobel Edge Feature Extraction**. Its job is to recover edge and texture cues directly from the raw SEM image before the encoder compresses them. [file:18]

### Inputs and outputs
- Input: raw SEM image `I` with shape `(B, 3, H, W)` or `(B, 1, H, W)` depending on your preprocessing. [file:18]
- Output: edge feature tensor `E` with shape `(B, 256, H/16, W/16)` so it matches the SAM latent resolution used for fusion. [file:18]

### Step-by-step algorithm
1. Convert the image to grayscale if needed. [file:18]
2. Apply fixed **Sobel-x** and **Sobel-y** kernels to get horizontal and vertical gradients. [file:18]
3. Compute Sobel magnitude: `sqrt(Gx^2 + Gy^2 + eps)`. [file:18]
4. Apply a bank of fixed **Gabor filters** at multiple orientations such as `0, 45, 90, 135` degrees. [file:18]
5. Aggregate the Gabor responses, usually by mean or concat across orientations. [file:18]
6. Concatenate grayscale, Sobel magnitude, and aggregated Gabor response into a small edge stack. [file:18]
7. Pass that stack through a shallow trainable CNN with stride-2 downsamples until you reach the latent spatial resolution. [file:18]
8. Project to 256 channels. [file:18]

### Pseudocode
```text
Input: image I
gray = to_gray(I)
sobel_x = conv(gray, K_sobel_x)
sobel_y = conv(gray, K_sobel_y)
sobel_mag = sqrt(sobel_x^2 + sobel_y^2 + eps)

for each orientation theta in {0,45,90,135}:
    gabor_theta = conv(gray, K_gabor(theta))
stack all gabor_theta -> G
gabor_mean = mean(G over orientation)

X = concat(gray, sobel_mag, gabor_mean)
E = CNN_downsample_project(X)
return E
```


## MDFF algorithm

### Purpose
MDFF means **Multi-scale Denoised Feature Fusion**. It takes the four GA tensors from the encoder and denoises them with wavelet processing before merging them into one clean latent representation. [file:18]

### Inputs and outputs
- Input: `GA1, GA2, GA3, GA4`, the four hooked global-attention stage features. [file:18]
- Output: denoised fused feature tensor `D` with shape `(B, 256, H_latent, W_latent)`. [file:18]

### Core idea
Each GA level first gets projected to a common channel width, then decomposed by Haar wavelets into `LL, LH, HL, HH`. The detail bands are soft-thresholded to suppress noise, reconstructed back by inverse Haar transform, and finally all four levels are fused. [file:18]

### Step-by-step algorithm
1. Take the 4 GA feature maps from encoder hooks. [file:18]
2. If the hooked format is spatial-first `(B, H, W, C)`, permute to `(B, C, H, W)`. [file:18]
3. For each GA level, project its native channels into a common 256-channel space using a `1x1` conv. [file:18]
4. Resize every projected feature to the same latent size if needed. [file:18]
5. Apply single-level 2D Haar wavelet decomposition to obtain `LL, LH, HL, HH`. [file:18]
6. Keep `LL` mostly intact, because it stores low-frequency structure. [file:18]
7. Apply learnable soft-thresholding to `LH, HL, HH` to suppress noisy high-frequency components. [file:18]
8. Reconstruct each feature map with inverse Haar transform. [file:18]
9. Apply channel attention, such as squeeze-excite, to each denoised level. [file:18]
10. Concatenate the four denoised levels along channels. [file:18]
11. Fuse them with a small convolutional block back to 256 channels. [file:18]

### Pseudocode
```text
Input: GA = [GA1, GA2, GA3, GA4]
processed = []
for each GA_i in GA:
    X = project_1x1(GA_i)
    X = resize_to_common_resolution(X)

    LL, LH, HL, HH = haar_dwt(X)
    LH = soft_threshold(LH, t_i)
    HL = soft_threshold(HL, t_i)
    HH = soft_threshold(HH, t_i)
    X_hat = inverse_haar(LL, LH, HL, HH)

    X_hat = channel_attention(X_hat)
    append X_hat to processed

D = concat(processed)
D = conv_fusion(D)
return D
```


## Haar wavelet denoising details

The attached note describes MDFF using Haar DWT with four sub-bands and learnable soft-thresholding. [file:18]

For a feature map `X`, Haar decomposition produces:
- `LL`: low-low, coarse structure. [file:18]
- `LH`: low-high, horizontal detail. [file:18]
- `HL`: high-low, vertical detail. [file:18]
- `HH`: high-high, diagonal detail. [file:18]

Soft-thresholding is:

\[
\hat{x} = \mathrm{sign}(x) \cdot \max(|x| - t, 0)
\]

where `t` is a learned nonnegative threshold, usually one per channel. This removes small noisy coefficients while preserving strong structure. [file:18]


## End-to-end fusion algorithm

The attached architecture note says the final fusion combines the SAM mask-side feature, SAM output token, MDFF denoised feature, and GSEFE edge feature with point-wise products. [file:18]

A clean no-FiLM algorithm is:

1. Run raw image through frozen SAM encoder and collect GA1-GA4 by hooks. [file:18]
2. Run raw image through GSEFE to get `E`. [file:18]
3. Run GA1-GA4 through MDFF to get `D`. [file:18]
4. Run SAM decoder to get its mask-side latent feature `M` and output token `T`. [file:18]
5. Broadcast `T` spatially and combine it with `M`. [file:18]
6. Combine `M * T` with `D * E` using element-wise product or concat-plus-conv. [file:18]
7. Upsample through a light decoder to get final binary mask logits. [file:18]

### Compact pseudocode
```text
GA = hook_encoder_features(image)
E  = GSEFE(image)
D  = MDFF(GA)
M, T = SAM_decoder_features(image)
T = spatial_broadcast(T)
Z_sam = M * T
Z_lbms = D * E
Z = fuse(Z_sam, Z_lbms)
mask_logits = upsample_head(Z)
return mask_logits
```


## What to implement in `.py` files

### `gsefe.py`
Implement these components:
- Fixed Sobel kernels as registered buffers. [file:18]
- Fixed multi-orientation Gabor kernels as registered buffers. [file:18]
- Optional grayscale conversion layer. [file:18]
- Trainable enhancement/downsample CNN ending at 256 channels and latent spatial size. [file:18]

### `mdff.py`
Implement these components:
- Per-level `1x1` channel projection to 256 channels. [file:18]
- Haar DWT function. [file:18]
- Learnable soft-thresholding module. [file:18]
- Inverse Haar reconstruction. [file:18]
- Per-level channel attention. [file:18]
- Final concatenation and fusion conv block. [file:18]
